In [1]:
import pandas as pd, scipy.stats as stats
# replacing string values in pandas.
# $2,120.20
# df["col_name"] = df["col_name"].str.replace("$", "").str.replace(",","").astype(float)
# df = df.drop_na(subset=["col1", "col2"])

url = "https://raw.githubusercontent.com/dataprofessor/data/refs/heads/master/penguins_cleaned.csv"
# https://raw.githubusercontent.com/
# dataprofessor/data/refs/heads/master/penguins_cleaned.csv

df = pd.read_csv(url)
df.head(1)


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181,3750,male


In [2]:
# t-test: body_mass_g divided by sex
mp = df.loc[df['sex'] == "male", "body_mass_g"]
fp = df.loc[df['sex'] == "female", "body_mass_g"]
t, p = stats.ttest_ind(mp, fp)
print(f"t={t}, p={p:.4f}")

t=8.541720337994516, p=0.0000


In [3]:
alpha = 0.05
degf = mp.count() + fp.count() - 2 # n1 + n2 - 2
crit_t = stats.t.ppf(1 - alpha / 2, degf)
print(f"t={t}, crit_t = {crit_t}, p={p:.4f}")

t=8.541720337994516, crit_t = 1.9671567996108663, p=0.0000


In [4]:
df['species'].unique()

<StringArray>
['Adelie', 'Gentoo', 'Chinstrap']
Length: 3, dtype: str

In [5]:
# a = df.loc[df["species"] == 'Adelie', "body_mass_g"] 
# g = 
# c =
species = [df.loc[df["species"] == s, "body_mass_g"] for s in df['species'].unique()]
f, p = stats.f_oneway(*species)
print(f"f={f:.4f}, p={p:.4f}")

f=341.8949, p=0.0000


In [6]:
degfb = df['species'].nunique() - 1
degfw = df['body_mass_g'].count() - df['species'].nunique()
crit_f = stats.f.ppf(1-alpha, degfb, degfw)
print(f"f={f:.4f}, crit_f={crit_f:.4f}, p={p:.4f}")

f=341.8949, crit_f=3.0231, p=0.0000


In [7]:
# Tukey HSD in Scipy
species_names = df['species'].unique()
species = [df.loc[df["species"] == s, "body_mass_g"] for s in species_names]
scipy_tukey = stats.tukey_hsd(*species) #->([],[],[]) rather than ([[],[],[]])
print(species_names)
print(scipy_tukey)


<StringArray>
['Adelie', 'Gentoo', 'Chinstrap']
Length: 3, dtype: str
Pairwise Group Comparisons (95.0% Confidence Interval)
Comparison  Statistic  p-value  Lower CI  Upper CI
 (0 - 1)  -1386.273     0.000 -1520.255 -1252.290
 (0 - 2)    -26.924     0.916  -186.201   132.353
 (1 - 0)   1386.273     0.000  1252.290  1520.255
 (1 - 2)   1359.349     0.000  1194.430  1524.267
 (2 - 0)     26.924     0.916  -132.353   186.201
 (2 - 1)  -1359.349     0.000 -1524.267 -1194.430



In [11]:
# Tukey HSD in statsmodels
## Must `uv add statsmodels`
from statsmodels.stats.multicomp import pairwise_tukeyhsd

sm_tukey = pairwise_tukeyhsd(
    endog=df["body_mass_g"],
    groups=df["species"],
    alpha=alpha, # 0.05 defined above
)
# Display the result
sm_tukey.summary()


group1,group2,meandiff,p-adj,lower,upper,reject
Adelie,Chinstrap,26.9239,0.9164,-132.3528,186.2005,False
Adelie,Gentoo,1386.2726,0.0,1252.2897,1520.2554,True
Chinstrap,Gentoo,1359.3487,0.0,1194.4304,1524.2671,True


In [9]:
df.loc[df['species'] == "Gentoo", "body_mass_g"].mean() - df.loc[df['species'] == "Adelie", "body_mass_g"].mean()

np.float64(1386.2725912282726)